In [1]:
# 06_RESULT — PART1 / CELL 01
# CROSS-PHASE INTEGRITY & RESULTS BLUEPRINT
#
# SOURCE READ-ONLY | NO TRAINING | NO OVERWRITE
# 348-ID RECONCILIATION | 17 FIGURE HASH CHECK
# IMMUTABLE RESULTS + SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MOD = ROOT / "04_Modeling" / "outputs"
TMP = ROOT / "05_Temporal" / "outputs"
OUT = ROOT / "06_Result" / "outputs" / "cross_phase_integrity"

MODEL = MOD / "leakage_grouping" / "modeling_labeled_348_grouped.csv"
TEMPORAL = TMP / "timestamp_canonicalization" / "temporal_labeled_348_locked.csv"
OOF = MOD / "comparative_evaluation" / "paired_oof_comparison_348.csv"
METRICS = MOD / "comparative_evaluation" / "model_comparison_metrics.csv"
DAILY = TMP / "daily_aggregation" / "daily_sentiment_counts_20_days.csv"

MODEL_FINDINGS = MOD / "final_evaluation_report" / "final_research_findings.json"
TEMP_FINDINGS = TMP / "comparative_temporal_report" / "temporal_research_findings.json"
MODEL_AUDIT = MOD / "final_evaluation_report" / "part20_final_report_audit.json"
TEMP_AUDIT = TMP / "comparative_temporal_report" / "part5_comparative_audit.json"

MODEL_REG = MOD / "final_evaluation_report" / "final_figure_registry.csv"
TEMP_REG = TMP / "timeline_visualizations" / "temporal_figure_registry.csv"

MODEL_FIG_DIR = MOD / "lstm_visualizations"
TEMP_FIG_DIR = TMP / "timeline_visualizations"

RECON_OUT = OUT / "cross_phase_dataset_reconciliation_348.csv"
METRICS_OUT = OUT / "locked_model_metrics.csv"
FIGURES_OUT = OUT / "integrated_figure_registry_17.csv"
BLUEPRINT_OUT = OUT / "results_chapter_blueprint.md"
AUDIT_OUT = OUT / "part1_cross_phase_integrity_audit.json"

OUTPUTS = [
    RECON_OUT,
    METRICS_OUT,
    FIGURES_OUT,
    BLUEPRINT_OUT,
    AUDIT_OUT
]

EXPECTED = {
    MODEL: "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7",
    TEMPORAL: "4ade3c2d0832f929e0a70ec0ae921653abdcb2aecf715a76fe0c274e674b5267",
    OOF: "670a444614a124c1b80106aa9c733c0d6b1cb690fd0ab0f20c9841d431323b1a",
    METRICS: "712aa656e0e5cee36db6f11ac4c94b4e1295cb93f87faac71e57f521091e3c57",
    DAILY: "4e4e1fd9e8b4e25b1aef729905b81306dd55a0d8053a2b509cee5d8acf89df21",
}

CASE_COUNTS = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142
}
LABEL_COUNTS = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relative(path):
    return path.relative_to(ROOT).as_posix()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, obj):
    path.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8"
    )

def load_csv(path):
    return pd.read_csv(
        path, encoding="utf-8-sig",
        dtype={"id": "string"}, keep_default_na=False
    )

print("=" * 75)
print("06_RESULT PART1 — CROSS-PHASE PREFLIGHT")
print("=" * 75)

# ------------------------------------------------------------
# A. VERIFY LOCKED SOURCES
# ------------------------------------------------------------

SOURCES = list(EXPECTED) + [
    MODEL_FINDINGS, TEMP_FINDINGS,
    MODEL_AUDIT, TEMP_AUDIT,
    MODEL_REG, TEMP_REG
]

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing output; refusing overwrite: {path}"
        )

source_before = {
    relative(p): file_sha(p)
    for p in SOURCES
}

for path, expected_sha in EXPECTED.items():
    assert file_sha(path) == expected_sha, (
        f"Locked SHA256 mismatch: {relative(path)}"
    )

model_audit = read_json(MODEL_AUDIT)
temporal_audit = read_json(TEMP_AUDIT)
model_findings = read_json(MODEL_FINDINGS)
temporal_findings = read_json(TEMP_FINDINGS)

assert model_audit["status"] == "PART20_FINAL_REPORT_AUDIT_PASS"
assert temporal_audit["status"] == "TEMPORAL_PART5_COMPARATIVE_AUDIT_PASS"
assert model_findings["dataset_rows"] == 348
assert temporal_findings["sample_size"] == 348
assert temporal_findings["comparison_type"] == "descriptive"

print("LOCKED SOURCE HASHES AND FINAL AUDITS: PASS")

# ------------------------------------------------------------
# B. RECONCILE ALL 348 IDs, CASES, LABELS, GROUPS
# ------------------------------------------------------------

model = load_csv(MODEL)
temporal = load_csv(TEMPORAL)
oof = load_csv(OOF)

assert len(model) == len(temporal) == len(oof) == 348

for df, name in [
    (model, "model"),
    (temporal, "temporal"),
    (oof, "oof")
]:
    assert df["id"].notna().all(), f"Missing ID: {name}"
    assert df["id"].is_unique, f"Duplicate IDs: {name}"

assert model["case"].value_counts().to_dict() == CASE_COUNTS
assert temporal["case"].value_counts().to_dict() == CASE_COUNTS
assert model["sentiment_label"].value_counts().to_dict() == LABEL_COUNTS
assert temporal["sentiment_label"].value_counts().to_dict() == LABEL_COUNTS

base = model[[
    "id", "case", "sentiment_label", "leakage_group_id"
]].copy()

base = base.rename(columns={
    "case": "case_model",
    "sentiment_label": "label_model",
    "leakage_group_id": "group_model"
})

temp_sub = temporal[[
    "id", "case", "sentiment_label",
    "leakage_group_id", "date_wib"
]].rename(columns={
    "case": "case_temporal",
    "sentiment_label": "label_temporal",
    "leakage_group_id": "group_temporal"
})

oof_sub = oof[[
    "id", "case", "true_label",
    "leakage_group_id", "validation_fold",
    "predicted_label_baseline", "predicted_label_lstm"
]].rename(columns={
    "case": "case_oof",
    "true_label": "label_oof",
    "leakage_group_id": "group_oof"
})

recon = base.merge(
    temp_sub, on="id", how="outer",
    indicator="merge_model_temporal",
    validate="one_to_one"
)

recon = recon.merge(
    oof_sub, on="id", how="outer",
    indicator="merge_with_oof",
    validate="one_to_one"
)

assert len(recon) == 348
assert recon["merge_model_temporal"].eq("both").all()
assert recon["merge_with_oof"].eq("both").all()

checks = {
    "case_match": ("case_model", "case_temporal", "case_oof"),
    "label_match": ("label_model", "label_temporal", "label_oof"),
    "group_match": ("group_model", "group_temporal", "group_oof")
}

for check_name, columns in checks.items():
    recon[check_name] = (
        recon[columns[0]].astype(str).eq(
            recon[columns[1]].astype(str)
        )
        &
        recon[columns[0]].astype(str).eq(
            recon[columns[2]].astype(str)
        )
    )
    assert recon[check_name].all(), (
        f"Cross-phase mismatch: {check_name}"
    )

assert recon["date_wib"].astype(str).str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()

assert recon["validation_fold"].nunique() == 3

assert recon["predicted_label_baseline"].isin(
    LABEL_COUNTS
).all()

assert recon["predicted_label_lstm"].isin(
    LABEL_COUNTS
).all()

recon = recon.sort_values(
    ["case_model", "id"],
    kind="mergesort"
).reset_index(drop=True)

print("348-ID MODELING/TEMPORAL/OOF JOIN: PASS")
print("CASE, LABEL, LEAKAGE GROUP RECONCILIATION: PASS")

# ------------------------------------------------------------
# C. VERIFY MODEL METRICS
# ------------------------------------------------------------

metrics = pd.read_csv(METRICS, encoding="utf-8-sig")

assert len(metrics) == 2
assert set(metrics["model"]) == {"baseline", "lstm"}
assert (metrics["rows"] == 348).all()

by_model = metrics.set_index("model")

for name in ["baseline", "lstm"]:
    for key in ["accuracy", "macro_f1", "macro_recall"]:
        assert np.isclose(
            float(by_model.loc[name, key]),
            float(model_findings[name][key]),
            rtol=0, atol=1e-12
        ), f"Metric inconsistency: {name}/{key}"

# Independently reconstruct accuracy from OOF predictions.
for name, pred_col in [
    ("baseline", "predicted_label_baseline"),
    ("lstm", "predicted_label_lstm")
]:
    accuracy = (
        recon[pred_col].astype(str)
        == recon["label_model"].astype(str)
    ).mean()

    assert np.isclose(
        float(accuracy),
        float(by_model.loc[name, "accuracy"]),
        rtol=0, atol=1e-12
    )

assert (
    float(by_model.loc["baseline", "macro_f1"])
    > float(by_model.loc["lstm", "macro_f1"])
)

print("BASELINE/LSTM METRICS AND OOF ACCURACY: PASS")

# ------------------------------------------------------------
# D. RECONCILE TEMPORAL AGGREGATION
# ------------------------------------------------------------

daily = pd.read_csv(DAILY, encoding="utf-8-sig")

assert len(daily) == 20
assert int(daily["total_tweets"].sum()) == 348

for case, expected in CASE_COUNTS.items():
    part = daily.loc[daily["case"] == case]
    assert int(part["total_tweets"].sum()) == expected

    matching = recon.loc[recon["case_model"] == case]
    assert len(matching) == expected

    for label in LABEL_COUNTS:
        actual = int(part[f"count_{label}"].sum())
        expected_count = int(
            (matching["label_model"] == label).sum()
        )
        assert actual == expected_count, (
            f"Temporal mismatch: {case}/{label}"
        )

# Exact day/label match, not just total counts.
daily_from_tweets = (
    recon.groupby(
        ["case_model", "date_wib", "label_model"]
    ).size()
)

for row in daily.itertuples(index=False):
    for label in LABEL_COUNTS:
        expected_count = int(daily_from_tweets.get(
            (row.case, row.date_wib, label), 0
        ))
        assert int(getattr(row, f"count_{label}")) == expected_count

print("DAILY SENTIMENT RECONCILIATION: PASS")

# ------------------------------------------------------------
# E. VERIFY AND INTEGRATE 17 EXISTING FIGURES
# ------------------------------------------------------------

model_registry = pd.read_csv(
    MODEL_REG, encoding="utf-8-sig"
)
temp_registry = pd.read_csv(
    TEMP_REG, encoding="utf-8-sig"
)

assert len(model_registry) == 9
assert len(temp_registry) == 8

figure_rows = []
figure_sources = []

for row in model_registry.itertuples(index=False):
    path = MODEL_FIG_DIR / row.filename

    assert path.is_file(), f"Missing figure: {path}"
    assert file_sha(path) == row.sha256
    assert path.open("rb").read(8) == bytes.fromhex(
        "89504e470d0a1a0a"
    )

    figure_sources.append(path)
    figure_rows.append({
        "integrated_figure_id": f"MODEL_{row.figure_id}",
        "stage": "04_Modeling",
        "filename": row.filename,
        "relative_repository_path": relative(path),
        "sha256": file_sha(path),
        "usage": "modeling_results"
    })

for row in temp_registry.itertuples(index=False):
    path = TEMP_FIG_DIR / row.filename

    assert path.is_file(), f"Missing figure: {path}"
    assert file_sha(path) == row.figure_sha256
    assert path.open("rb").read(8) == bytes.fromhex(
        "89504e470d0a1a0a"
    )

    figure_sources.append(path)
    figure_rows.append({
        "integrated_figure_id": f"TEMP_{row.figure_id}",
        "stage": "05_Temporal",
        "filename": row.filename,
        "relative_repository_path": relative(path),
        "sha256": file_sha(path),
        "usage": "temporal_results"
    })

figure_registry = pd.DataFrame(figure_rows)

assert len(figure_registry) == 17
assert figure_registry["integrated_figure_id"].is_unique

figure_before = {
    relative(p): file_sha(p) for p in figure_sources
}

print("9 MODELING + 8 TEMPORAL FIGURE HASHES: PASS")

# ------------------------------------------------------------
# F. RESULTS CHAPTER BLUEPRINT
# ------------------------------------------------------------

blueprint = """# BLUEPRINT BAB IV — HASIL DAN PEMBAHASAN

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Status:** Blueprint awal berdasarkan hasil audit lintas fase.

> Struktur ini dapat disesuaikan dengan proposal dan
> pedoman penulisan kampus. Belum merupakan naskah
> BAB IV final.

## 4.1 Gambaran Umum Dataset

- Jelaskan sumber dan proses seleksi data.
- Sajikan 348 tweet berlabel manusia.
- Distribusi kasus: Lombok 101, Palu 105, Sumatera 142.
- Distribusi sentimen: NETRAL 225, POSITIF 108, NEGATIF 15.
- Jelaskan 337 leakage groups dan pengendalian leakage.
- Jelaskan keterbatasan cakupan sampel.

## 4.2 Metode dan Pengaturan Evaluasi

- Stratified Group K-Fold tiga fold, seed 123.
- Baseline TF-IDF + Logistic Regression.
- Model LSTM_SMALL_V1.
- Metrik utama Out-of-Fold Macro-F1.
- Bedakan training, validation, dan OOF.
- Tidak ada independent holdout test set.

## 4.3 Hasil Pemodelan Sentimen

- Sajikan metrik kedua model.
- Baseline: accuracy 0.7040, Macro-F1 0.5236.
- LSTM: accuracy 0.5862, Macro-F1 0.4751.
- Bahas grafik hasil modeling dan confusion matrix.
- Jelaskan indikasi overfitting LSTM.
- Bahas kelemahan kedua model pada kelas NEGATIF.

## 4.4 Analisis Kesalahan Klasifikasi

- Kedua model benar: 165 tweet.
- Hanya baseline benar: 80 tweet.
- Hanya LSTM benar: 39 tweet.
- Kedua model salah: 64 tweet.
- Jelaskan perbedaan precision dan recall NEGATIF.
- Jangan menyatakan signifikansi statistik tanpa pengujian.

## 4.5 Distribusi Sentimen dan Analisis Temporal

- Gunakan label manusia, bukan prediksi model.
- Sajikan jumlah dan persentase harian per kasus.
- Tampilkan 20 hari kalender kasus.
- Sebanyak 15 hari memiliki observasi dan 5 hari kosong.
- Gunakan grafik temporal asli Part4.
- Hari relatif 0 berarti awal observasi, bukan awal bencana.

## 4.6 Pembahasan Perbandingan Kasus

- Bandingkan Lombok, Palu, dan Sumatera secara deskriptif.
- Bedakan jumlah tweet dan proporsi sentimen.
- Jelaskan variasi cakupan waktu dan distribusi sampel.
- Hindari generalisasi sebagai opini seluruh masyarakat.
- Hindari klaim hubungan sebab-akibat kebijakan.

## 4.7 Keterbatasan Penelitian

- Hanya 348 tweet berlabel.
- Kelas NEGATIF sangat kecil (15 tweet).
- Perbedaan jendela pengamatan antarkasus.
- Pengaruh strategi scraping dan seleksi sampel.
- Tidak tersedia independent holdout.
- Potensi overfitting pada LSTM.
- Hasil temporal bersifat deskriptif.

## 4.8 Ringkasan Temuan

- Baseline lebih tinggi menurut OOF Macro-F1.
- LSTM memiliki Macro-Recall lebih tinggi.
- NETRAL mendominasi sampel ketiga kasus.
- Pola temporal berbeda menurut kasus.
- Simpulan terbatas pada sampel dan metode penelitian.

## Catatan Integrasi

Modeling menjelaskan performa prediksi.
Temporal menjelaskan distribusi label manusia.
Keduanya menggunakan 348 tweet yang sama,
tetapi menjawab pertanyaan analitis berbeda.

Dokumen ini merupakan blueprint.
Finalisasi BAB IV harus disesuaikan dengan
rumusan masalah, tujuan, serta pedoman akademik.
"""

# ------------------------------------------------------------
# G. AUDIT MANIFEST
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

audit = {
    "stage": "06_Result",
    "part": "1 Cross-Phase Integrity & Blueprint",
    "generated_at_utc": now,
    "modeling_source_commit":
        "5cf80cc802f99c25fc4b714a4036beea3ae930e4",
    "temporal_source_commit":
        "7571fc02d68cdb2f81b404554df4304940e2bb07",
    "dataset_rows": 348,
    "unique_ids": 348,
    "case_match_count": int(recon["case_match"].sum()),
    "label_match_count": int(recon["label_match"].sum()),
    "leakage_group_match_count":
        int(recon["group_match"].sum()),
    "oof_rows": 348,
    "daily_rows": 20,
    "daily_tweets": 348,
    "model_figures_verified": 9,
    "temporal_figures_verified": 8,
    "total_figures_verified": 17,
    "source_sha256": source_before,
    "figure_sha256": figure_before,
    "model_training_performed": False,
    "source_modified": False,
    "results_blueprint_prepared": True,
    "output_sha256": {},
    "status": "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
}

# ------------------------------------------------------------
# H. IMMUTABLE EXPORT
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing output: {path}"
        )

token = uuid.uuid4().hex[:10]
temps = {
    p: OUT / f".result_p1_{token}_{i}.tmp"
    for i, p in enumerate(OUTPUTS)
}
created = []

try:
    recon.to_csv(
        temps[RECON_OUT],
        index=False, encoding="utf-8-sig"
    )

    metrics.to_csv(
        temps[METRICS_OUT],
        index=False, encoding="utf-8-sig"
    )

    figure_registry.to_csv(
        temps[FIGURES_OUT],
        index=False, encoding="utf-8-sig"
    )

    temps[BLUEPRINT_OUT].write_text(
        blueprint, encoding="utf-8"
    )

    audit["output_sha256"] = {
        p.name: file_sha(temps[p])
        for p in OUTPUTS[:-1]
    }

    write_json(temps[AUDIT_OUT], audit)

    # Check no source changed before publication.
    assert {
        relative(p): file_sha(p)
        for p in SOURCES
    } == source_before

    assert {
        relative(p): file_sha(p)
        for p in figure_sources
    } == figure_before

    # Re-open saved outputs and verify row counts.
    assert len(pd.read_csv(temps[RECON_OUT])) == 348
    assert len(pd.read_csv(temps[FIGURES_OUT])) == 17

    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(temps[destination].read_bytes())
        created.append(destination)

except Exception:
    # Remove only outputs created by this run.
    for path in created:
        if path.exists():
            path.unlink()
    raise

finally:
    for path in temps.values():
        if path.exists():
            path.unlink()

# ------------------------------------------------------------
# I. FINAL VERIFICATION
# ------------------------------------------------------------

for name, expected_sha in audit["output_sha256"].items():
    assert file_sha(OUT / name) == expected_sha

assert {
    relative(p): file_sha(p)
    for p in SOURCES
} == source_before

assert {
    relative(p): file_sha(p)
    for p in figure_sources
} == figure_before

assert read_json(AUDIT_OUT)["status"] == (
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
)

print("\n" + "=" * 75)
print("06_RESULT PART1 — FINAL AUDIT")
print("=" * 75)

print("Cross-phase rows     :", len(recon))
print("Matching IDs         :", recon["id"].nunique())
print("Matching cases       :", int(recon["case_match"].sum()))
print("Matching labels      :", int(recon["label_match"].sum()))
print("Matching groups      :", int(recon["group_match"].sum()))
print("Modeling figures     :", 9)
print("Temporal figures     :", 8)
print("Total figures        :", len(figure_registry))
print("Source modified      : NO")
print("Training performed   : NO")

print("\nLOCKED MODEL METRICS")
display(metrics)

print("\nINTEGRITY CHECK COUNTS")
display(
    recon[[
        "case_match", "label_match", "group_match"
    ]].sum().to_frame("matching_rows")
)

print("\nARTIFACTS:")
for p in OUTPUTS:
    print("-", p)

print("\nFINAL STATUS:", audit["status"])
print("=" * 75)


06_RESULT PART1 — CROSS-PHASE PREFLIGHT
LOCKED SOURCE HASHES AND FINAL AUDITS: PASS
348-ID MODELING/TEMPORAL/OOF JOIN: PASS
CASE, LABEL, LEAKAGE GROUP RECONCILIATION: PASS
BASELINE/LSTM METRICS AND OOF ACCURACY: PASS
DAILY SENTIMENT RECONCILIATION: PASS
9 MODELING + 8 TEMPORAL FIGURE HASHES: PASS

06_RESULT PART1 — FINAL AUDIT
Cross-phase rows     : 348
Matching IDs         : 348
Matching cases       : 348
Matching labels      : 348
Matching groups      : 348
Modeling figures     : 9
Temporal figures     : 8
Total figures        : 17
Source modified      : NO
Training performed   : NO

LOCKED MODEL METRICS


,model,rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,baseline,348,0.704023,0.776108,0.494321,0.523555,0.689845
1,lstm,348,0.586207,0.475956,0.559630,0.475130,0.617452



INTEGRITY CHECK COUNTS


,matching_rows
case_match,348
label_match,348
group_match,348



ARTIFACTS:
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\cross_phase_dataset_reconciliation_348.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\locked_model_metrics.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\integrated_figure_registry_17.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\results_chapter_blueprint.md
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\part1_cross_phase_integrity_audit.json

FINAL STATUS: RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS


In [3]:
# 06_RESULT — PART2 / CELL 02
# CONSOLIDATED RESEARCH RESULTS
#
# LOCKED SOURCES | OOF METRIC RECONCILIATION
# DATASET + MODEL + ERROR + TEMPORAL TABLES
# NO TRAINING | NO OVERWRITE | SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
MOD = ROOT / "04_Modeling" / "outputs"
TEMP = ROOT / "05_Temporal" / "outputs"
P1 = ROOT / "06_Result" / "outputs" / "cross_phase_integrity"
OUT = ROOT / "06_Result" / "outputs" / "consolidated_results"

RECON = P1 / "cross_phase_dataset_reconciliation_348.csv"
METRICS = P1 / "locked_model_metrics.csv"
P1_AUDIT = P1 / "part1_cross_phase_integrity_audit.json"

OOF = MOD / "comparative_evaluation" / "paired_oof_comparison_348.csv"
MODEL_FINDINGS = MOD / "final_evaluation_report" / "final_research_findings.json"
TEMP_FINDINGS = TEMP / "comparative_temporal_report" / "temporal_research_findings.json"
DAILY = TEMP / "daily_aggregation" / "daily_sentiment_counts_20_days.csv"

DATA_OUT = OUT / "dataset_characteristics.csv"
MODEL_OUT = OUT / "consolidated_model_evaluation.csv"
ERROR_OUT = OUT / "model_error_analysis_by_class.csv"
TEMP_OUT = OUT / "consolidated_temporal_findings.csv"
REPORT_OUT = OUT / "consolidated_research_results.md"
AUDIT_OUT = OUT / "part2_consolidated_results_audit.json"

OUTPUTS = [
    DATA_OUT, MODEL_OUT, ERROR_OUT,
    TEMP_OUT, REPORT_OUT, AUDIT_OUT
]

SOURCES = [
    RECON, METRICS, P1_AUDIT,
    OOF, MODEL_FINDINGS, TEMP_FINDINGS, DAILY
]

CASES = ["lombok_2018", "palu_2018", "sumatra_2025"]
LABELS = ["NETRAL", "POSITIF", "NEGATIF"]

CASE_NAMES = {
    "lombok_2018": "Lombok 2018",
    "palu_2018": "Palu 2018",
    "sumatra_2025": "Sumatera 2025"
}

EXPECTED_CASE = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142
}
EXPECTED_LABEL = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, value):
    path.write_text(
        json.dumps(
            value, indent=2,
            ensure_ascii=False, allow_nan=False
        ),
        encoding="utf-8"
    )

def read_csv(path, has_id=False):
    kwargs = {"encoding": "utf-8-sig"}
    if has_id:
        kwargs["dtype"] = {"id": "string"}
    return pd.read_csv(path, **kwargs)

def check_equal(a, b, name, atol=1e-12):
    assert np.isclose(
        float(a), float(b),
        atol=atol, rtol=0
    ), f"{name}: {a} != {b}"

def markdown_table(df, digits=4):
    """Convert DataFrame to Markdown without tabulate."""
    def fmt(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return f"{value:.{digits}f}"
        return str(value).replace("|", "\\|").replace("\n", " ")

    headers = [str(col) for col in df.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]

    for row in df.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(fmt(v) for v in row) + " |")

    return "\n".join(lines)

print("=" * 75)
print("RESULT PART2 — CONSOLIDATION PREFLIGHT")
print("=" * 75)

# ------------------------------------------------------------
# A. PREFLIGHT & SOURCE LOCK
# ------------------------------------------------------------

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing Part2 output; refusing overwrite: {path}"
        )

source_hashes = {
    p.relative_to(ROOT).as_posix(): sha256(p)
    for p in SOURCES
}

p1_audit = read_json(P1_AUDIT)
model_findings = read_json(MODEL_FINDINGS)
temp_findings = read_json(TEMP_FINDINGS)

assert p1_audit["status"] == (
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
)

for path in [RECON, METRICS]:
    expected = p1_audit["output_sha256"][path.name]
    assert sha256(path) == expected

assert sha256(OOF) == p1_audit["source_sha256"][
    "04_Modeling/outputs/comparative_evaluation/"
    "paired_oof_comparison_348.csv"
]

assert sha256(DAILY) == p1_audit["source_sha256"][
    "05_Temporal/outputs/daily_aggregation/"
    "daily_sentiment_counts_20_days.csv"
]

assert model_findings["status"] == (
    "PART20_FINAL_FINDINGS_PREPARED"
)
assert temp_findings["status"] == (
    "TEMPORAL_PART5_FINDINGS_PREPARED"
)

print("PART1 AUDIT & SOURCE SHA256: PASS")

# ------------------------------------------------------------
# B. LOAD & CHECK INTEGRATED TWEET DATA
# ------------------------------------------------------------

recon = read_csv(RECON, has_id=True)
oof = read_csv(OOF, has_id=True)
locked = read_csv(METRICS)
daily = read_csv(DAILY)

assert len(recon) == len(oof) == 348
assert recon["id"].is_unique
assert oof["id"].is_unique
assert len(daily) == 20

assert recon["case_match"].all()
assert recon["label_match"].all()
assert recon["group_match"].all()

assert set(recon["id"]) == set(oof["id"])

check = recon[[
    "id", "label_model",
    "predicted_label_baseline",
    "predicted_label_lstm"
]].merge(
    oof[[
        "id", "true_label",
        "predicted_label_baseline",
        "predicted_label_lstm"
    ]],
    on="id", validate="one_to_one",
    suffixes=("_recon", "_oof")
)

assert (check["label_model"] == check["true_label"]).all()

for model in ["baseline", "lstm"]:
    assert (
        check[f"predicted_label_{model}_recon"]
        == check[f"predicted_label_{model}_oof"]
    ).all()

assert recon["case_model"].value_counts().to_dict() == (
    EXPECTED_CASE
)
assert recon["label_model"].value_counts().to_dict() == (
    EXPECTED_LABEL
)

print("348 TWEET-LEVEL PREDICTIONS: PASS")

# ------------------------------------------------------------
# C. DATASET CHARACTERISTICS TABLE
# ------------------------------------------------------------

dataset_rows = []

for case in CASES:
    subset = recon[recon["case_model"] == case]
    n = len(subset)

    row = {
        "case": case,
        "case_name": CASE_NAMES[case],
        "total_tweets": n,
        "sample_pct": n / len(recon) * 100,
        "unique_leakage_groups":
            subset["group_model"].nunique()
    }

    for label in LABELS:
        count = int((subset["label_model"] == label).sum())
        row[f"count_{label}"] = count
        row[f"pct_{label}"] = count / n * 100

    dataset_rows.append(row)

dataset_table = pd.DataFrame(dataset_rows)

assert dataset_table["total_tweets"].sum() == 348

for label in LABELS:
    assert int(dataset_table[f"count_{label}"].sum()) == (
        EXPECTED_LABEL[label]
    )

unique_groups = recon["group_model"].nunique()
assert unique_groups == model_findings["unique_leakage_groups"]
assert unique_groups == 337

print("DATASET CHARACTERISTICS: PASS")

# ------------------------------------------------------------
# D. RECOMPUTE MODEL PERFORMANCE FROM OOF
# ------------------------------------------------------------

locked = locked.set_index("model")
assert set(locked.index) == {"baseline", "lstm"}

model_rows = []
class_rows = []

y_true = oof["true_label"].astype(str).to_numpy()

for model in ["baseline", "lstm"]:

    y_pred = (
        oof[f"predicted_label_{model}"]
        .astype(str).to_numpy()
    )

    accuracy = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    )

    weighted_p, weighted_r, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true, y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0
        )
    )

    results = {
        "model": model,
        "n_oof": len(oof),
        "accuracy": float(accuracy),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "evaluation": "3-fold SGKF OOF",
        "independent_holdout": False
    }

    for metric in [
        "accuracy", "macro_precision",
        "macro_recall", "macro_f1", "weighted_f1"
    ]:
        check_equal(
            results[metric],
            locked.loc[model, metric],
            f"{model}/{metric}"
        )

    model_rows.append(results)

    class_p, class_r, class_f, support = (
        precision_recall_fscore_support(
            y_true, y_pred,
            labels=LABELS,
            average=None,
            zero_division=0
        )
    )

    for i, label in enumerate(LABELS):
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true == label) & (y_pred != label)).sum())
        tn = int(((y_true != label) & (y_pred != label)).sum())

        assert tp + fn == int(support[i])
        assert tp + fp + fn + tn == 348

        class_rows.append({
            "model": model,
            "sentiment_label": label,
            "support": int(support[i]),
            "true_positive": tp,
            "false_positive": fp,
            "false_negative": fn,
            "true_negative": tn,
            "precision": float(class_p[i]),
            "recall": float(class_r[i]),
            "f1": float(class_f[i])
        })

model_table = pd.DataFrame(model_rows)
class_table = pd.DataFrame(class_rows)

assert len(model_table) == 2
assert len(class_table) == 6

assert (
    model_table.set_index("model").loc["baseline", "macro_f1"]
    >
    model_table.set_index("model").loc["lstm", "macro_f1"]
)

# Compare negative-class findings to Part20.
for model in ["baseline", "lstm"]:
    neg = class_table[
        (class_table["model"] == model)
        & (class_table["sentiment_label"] == "NEGATIF")
    ].iloc[0]

    reference = model_findings["negative_class"][model]

    assert int(neg["true_positive"]) == reference["true_positive"]
    assert int(neg["false_positive"]) == reference["false_positive"]
    assert int(neg["false_negative"]) == reference["false_negative"]

    for metric in ["precision", "recall", "f1"]:
        check_equal(
            neg[metric],
            reference[metric],
            f"NEGATIF/{model}/{metric}"
        )

print("OOF METRICS & NEGATIVE CLASS RECOMPUTATION: PASS")

# ------------------------------------------------------------
# E. PAIRED ERROR ANALYSIS
# ------------------------------------------------------------

baseline_correct = (
    oof["true_label"] == oof["predicted_label_baseline"]
)
lstm_correct = (
    oof["true_label"] == oof["predicted_label_lstm"]
)

paired = {
    "both_correct":
        int((baseline_correct & lstm_correct).sum()),
    "baseline_only_correct":
        int((baseline_correct & ~lstm_correct).sum()),
    "lstm_only_correct":
        int((~baseline_correct & lstm_correct).sum()),
    "both_wrong":
        int((~baseline_correct & ~lstm_correct).sum())
}

assert sum(paired.values()) == 348
assert paired == model_findings["paired_correctness"]

print("PAIRED OOF ERROR CATEGORY COUNTS: PASS")

# ------------------------------------------------------------
# F. CONSOLIDATE TEMPORAL FINDINGS
# ------------------------------------------------------------

temp_rows = []

for finding in temp_findings["case_findings"]:
    case = finding["case"]
    subset = daily[daily["case"] == case]

    count = int(subset["total_tweets"].sum())
    observed_days = int((subset["total_tweets"] > 0).sum())
    zero_days = int((subset["total_tweets"] == 0).sum())

    assert count == finding["total_tweets"]
    assert observed_days == finding["observed_days"]
    assert zero_days == finding["zero_tweet_days"]
    assert len(subset) == finding["calendar_days"]

    peak = int(subset["total_tweets"].max())
    peak_dates = sorted(
        subset.loc[
            subset["total_tweets"] == peak, "date_wib"
        ].astype(str).tolist()
    )

    assert peak == finding["peak_daily_tweet_count"]
    assert peak_dates == sorted(finding["peak_dates_wib"])

    record = {
        "case": case,
        "case_name": CASE_NAMES[case],
        "total_tweets": count,
        "first_date_wib": subset["date_wib"].min(),
        "last_date_wib": subset["date_wib"].max(),
        "calendar_days": len(subset),
        "observed_days": observed_days,
        "zero_tweet_days": zero_days,
        "peak_date_wib": "; ".join(peak_dates),
        "peak_daily_tweets": peak,
        "peak_share_pct": peak / count * 100
    }

    for label in LABELS:
        c = int(subset[f"count_{label}"].sum())

        assert c == finding["sentiment_counts"][label]

        record[f"count_{label}"] = c
        record[f"pct_{label}"] = c / count * 100

    temp_rows.append(record)

temp_table = pd.DataFrame(temp_rows)

assert len(temp_table) == 3
assert temp_table["total_tweets"].sum() == 348
assert temp_table["calendar_days"].sum() == 20
assert temp_table["observed_days"].sum() == 15
assert temp_table["zero_tweet_days"].sum() == 5

print("TEMPORAL FINDINGS RECONCILIATION: PASS")

# ------------------------------------------------------------
# G. PREPARE INTEGRATED RESEARCH REPORT
# ------------------------------------------------------------

model_view = model_table[[
    "model", "n_oof", "accuracy",
    "macro_precision", "macro_recall",
    "macro_f1", "weighted_f1"
]].copy()

data_view = dataset_table[[
    "case_name", "total_tweets",
    "count_NETRAL", "count_POSITIF", "count_NEGATIF"
]].copy()

error_view = class_table[[
    "model", "sentiment_label", "support",
    "true_positive", "false_positive",
    "false_negative", "precision", "recall", "f1"
]].copy()

temp_view = temp_table[[
    "case_name", "calendar_days",
    "observed_days", "zero_tweet_days",
    "peak_date_wib", "peak_daily_tweets"
]].copy()

b = model_table.set_index("model").loc["baseline"]
l = model_table.set_index("model").loc["lstm"]

report = f"""# HASIL PENELITIAN TERKONSOLIDASI

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Tahap:** 06_Result Part2 — Consolidated Research Results

**Tanggal pembuatan UTC:** {datetime.now(timezone.utc).isoformat()}

## 1. Ruang Lingkup

Laporan ini mengintegrasikan hasil fase Modeling dan
Temporal menggunakan 348 tweet yang telah direkonsiliasi
pada Result Part1.

Perhitungan model bersumber dari prediksi out-of-fold (OOF).
Analisis temporal menggunakan label manusia yang dikunci.

Tidak dilakukan pelatihan model atau penambahan data baru.

## 2. Karakteristik Dataset

{markdown_table(data_view)}

Dataset berisi **348 tweet**, dengan distribusi label:
NETRAL **225**, POSITIF **108**, dan NEGATIF **15**.

Jumlah leakage group unik: **{unique_groups}**.
Distribusi kelas tidak seimbang; karena itu interpretasi
metrik tidak cukup hanya menggunakan accuracy.

## 3. Evaluasi Model

{markdown_table(model_view)}

Protokol evaluasi: **{model_findings["cv_configuration"]}**.
Metrik utama: **{model_findings["primary_metric"]}**.

Baseline TF-IDF + Logistic Regression memiliki
OOF Macro-F1 **{b["macro_f1"]:.4f}**, sedangkan
LSTM memiliki **{l["macro_f1"]:.4f}**.

Perbedaan Macro-F1 baseline dikurangi LSTM adalah
**{b["macro_f1"] - l["macro_f1"]:.4f}** poin.

Baseline juga memiliki accuracy yang lebih tinggi.
Sebaliknya, Macro-Recall LSTM lebih tinggi.

Peringkat tersebut bersifat deskriptif untuk hasil OOF.
Tidak ada independent holdout test set.

## 4. Analisis Kesalahan per Kelas

{markdown_table(error_view)}

Kelas NEGATIF memiliki 15 contoh aktual.
Baseline menghasilkan recall NEGATIF lebih rendah,
sementara LSTM memiliki lebih banyak false positive
NEGATIF. Perbedaan ini harus dibahas sebagai
trade-off precision dan recall, bukan hanya jumlah
prediksi benar.

## 5. Perbandingan Kesalahan Berpasangan

| Kategori | Jumlah tweet |
| --- | ---: |
| Kedua model benar | {paired["both_correct"]} |
| Hanya baseline benar | {paired["baseline_only_correct"]} |
| Hanya LSTM benar | {paired["lstm_only_correct"]} |
| Kedua model salah | {paired["both_wrong"]} |
| **Total** | **348** |

Perbandingan ini menunjukkan tumpang tindih dan
perbedaan pola kesalahan kedua model pada tweet
yang sama. Belum dilakukan uji signifikansi statistik.

## 6. Ringkasan Hasil Temporal

{markdown_table(temp_view)}

Jumlah kalender kasus: **20 hari**.
Hari dengan observasi: **15 hari**.
Hari tanpa observasi: **5 hari**.

Komposisi sentimen NETRAL merupakan yang terbesar
pada ketiga kasus. Hari tanpa tweet ditandai secara
eksplisit dan tidak dihitung sebagai persentase nol.

Perbandingan volume dan proporsi bersifat deskriptif
atas sampel yang terkumpul, bukan estimasi aktivitas
atau opini seluruh pengguna X.

## 7. Sintesis Hasil

1. Baseline mengungguli LSTM pada metrik utama
   OOF Macro-F1, tetapi LSTM memiliki Macro-Recall
   lebih tinggi.
2. Kelas NEGATIF menjadi tantangan evaluasi
   karena support sangat kecil.
3. Analisis Temporal menggambarkan dominasi NETRAL
   pada ketiga kasus dan cakupan observasi yang
   tidak seragam.
4. Hasil Modeling mengevaluasi kemampuan prediksi,
   sementara hasil Temporal menggambarkan label
   manusia menurut waktu.
5. Tidak ada bukti dalam analisis ini yang cukup
   untuk menyimpulkan hubungan kausal antara
   penetapan status bencana dan sentimen masyarakat.

## 8. Batasan Interpretasi

- Sampel penelitian terdiri atas 348 tweet.
- Label NEGATIF hanya mencakup 15 tweet.
- Evaluasi model menggunakan OOF tiga fold,
  tanpa test set independen.
- Periode pengamatan antarkasus berbeda.
- Hari tanpa tweet tidak berarti aktivitas publik nol.
- Tidak dilakukan pengujian signifikansi atas
  perbedaan model dalam tahap ini.
- Tidak dilakukan inferensi populasi atau kausal.

## 9. Kesiapan Part3

Empat tabel konsolidasi dalam laporan ini
dapat digunakan untuk menyeleksi figur yang
relevan bagi BAB IV.

**Tahap berikutnya: 06_Result Part3 —
Results Visualization & Figure Selection.**
"""

# ------------------------------------------------------------
# H. CREATE OUTPUTS WITHOUT OVERWRITING
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite: {path}"
        )

output_bytes = {}

csv_tables = {
    DATA_OUT: dataset_table,
    MODEL_OUT: model_table,
    ERROR_OUT: class_table,
    TEMP_OUT: temp_table
}

for path, table in csv_tables.items():
    output_bytes[path] = table.to_csv(
        index=False
    ).encode("utf-8-sig")

output_bytes[REPORT_OUT] = report.encode("utf-8")

audit = {
    "stage": "06_Result",
    "part": "2 Consolidated Research Results",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "source_commit":
        "5f591e0456d380758ef4ac3f06253ca2eabe4be3",
    "source_sha256": source_hashes,
    "dataset_rows": 348,
    "unique_leakage_groups": unique_groups,
    "model_count": 2,
    "per_class_rows": 6,
    "case_count": 3,
    "daily_calendar_rows": 20,
    "paired_correctness": paired,
    "oof_metrics_recomputed": True,
    "oof_metrics_match_locked": True,
    "temporal_findings_recomputed": True,
    "source_modified": False,
    "model_training_performed": False,
    "output_sha256": {
        p.name: hashlib.sha256(data).hexdigest()
        for p, data in output_bytes.items()
    },
    "status": "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
}

output_bytes[AUDIT_OUT] = json.dumps(
    audit,
    ensure_ascii=False, indent=2,
    allow_nan=False
).encode("utf-8")

created = []

try:
    # Recheck all source hashes immediately before writing.
    assert {
        p.relative_to(ROOT).as_posix(): sha256(p)
        for p in SOURCES
    } == source_hashes

    for path in OUTPUTS:
        with path.open("xb") as f:
            f.write(output_bytes[path])
        created.append(path)

    # Verify complete output after writing.
    for path, original_bytes in output_bytes.items():
        assert sha256(path) == hashlib.sha256(
            original_bytes
        ).hexdigest()

    assert len(read_csv(DATA_OUT)) == 3
    assert len(read_csv(MODEL_OUT)) == 2
    assert len(read_csv(ERROR_OUT)) == 6
    assert len(read_csv(TEMP_OUT)) == 3

    assert read_json(AUDIT_OUT)["status"] == (
        "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
    )

    assert {
        p.relative_to(ROOT).as_posix(): sha256(p)
        for p in SOURCES
    } == source_hashes

except Exception:
    # Roll back only files created by this execution.
    for path in created:
        if path.exists():
            path.unlink()
    raise

# ------------------------------------------------------------
# I. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("06_RESULT PART2 — FINAL AUDIT")
print("=" * 75)

print("Dataset rows          : 348")
print("Leakage groups        :", unique_groups)
print("Models                :", len(model_table))
print("Per-class result rows :", len(class_table))
print("Disaster cases        :", len(temp_table))
print("Daily calendar rows   :", len(daily))
print("Source modified       : NO")
print("Model training        : NO")

print("\nMODEL EVALUATION:")
display(model_view.round(4))

print("\nPER-CLASS ERROR ANALYSIS:")
display(error_view.round(4))

print("\nTEMPORAL FINDINGS:")
display(temp_view)

print("\nOUTPUT FILES:")
for path in OUTPUTS:
    print("-", path.name)

print("\nFINAL STATUS:", audit["status"])
print("=" * 75)


RESULT PART2 — CONSOLIDATION PREFLIGHT
PART1 AUDIT & SOURCE SHA256: PASS
348 TWEET-LEVEL PREDICTIONS: PASS
DATASET CHARACTERISTICS: PASS
OOF METRICS & NEGATIVE CLASS RECOMPUTATION: PASS
PAIRED OOF ERROR CATEGORY COUNTS: PASS
TEMPORAL FINDINGS RECONCILIATION: PASS

06_RESULT PART2 — FINAL AUDIT
Dataset rows          : 348
Leakage groups        : 337
Models                : 2
Per-class result rows : 6
Disaster cases        : 3
Daily calendar rows   : 20
Source modified       : NO
Model training        : NO

MODEL EVALUATION:


,model,n_oof,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,baseline,348,0.7040,0.7761,0.4943,0.5236,0.6898
1,lstm,348,0.5862,0.4760,0.5596,0.4751,0.6175



PER-CLASS ERROR ANALYSIS:


,model,sentiment_label,support,true_positive,false_positive,false_negative,precision,recall,f1
0,baseline,NETRAL,225,187,62,38,0.7510,0.8311,0.7890
1,baseline,POSITIF,108,56,41,52,0.5773,0.5185,0.5463
2,baseline,NEGATIF,15,2,0,13,1.0000,0.1333,0.2353
3,lstm,NETRAL,225,139,36,86,0.7943,0.6178,0.6950
4,lstm,POSITIF,108,57,58,51,0.4957,0.5278,0.5112
5,lstm,NEGATIF,15,8,50,7,0.1379,0.5333,0.2192



TEMPORAL FINDINGS:


,case_name,calendar_days,observed_days,zero_tweet_days,peak_date_wib,peak_daily_tweets
0,Lombok 2018,9,6,3,2018-08-10,24
1,Palu 2018,5,5,0,2018-10-01,36
2,Sumatera 2025,6,4,2,2025-12-03,54



OUTPUT FILES:
- dataset_characteristics.csv
- consolidated_model_evaluation.csv
- model_error_analysis_by_class.csv
- consolidated_temporal_findings.csv
- consolidated_research_results.md
- part2_consolidated_results_audit.json

FINAL STATUS: RESULT_PART2_CONSOLIDATED_RESULTS_PASS
